In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("All libraries loaded successfully")

All libraries loaded successfully


In [3]:
from pathlib import Path

SCRIPT_DIR  = Path.cwd()
RAW_PATH    = SCRIPT_DIR.parent / "data" / "DataCoSupplyChainDataset.csv"
OUTPUT_PATH = SCRIPT_DIR.parent / "data"

df_raw = pd.read_csv(RAW_PATH, encoding='cp1252')

print(f"Raw data loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
print(df_raw.head())

Raw data loaded: 180,519 rows, 53 columns
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk  Category Id   Category Name Customer City  ...  \
0                   0           73  Sporting Goods        Caguas  ...   
1     

In [4]:

KEEP_COLUMNS = [
    'Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
    'Late_delivery_risk', 'Delivery Status', 'Shipping Mode',
    'Order Region', 'Order Status', 'Order Id', 'Order Item Id',
    'Order Item Quantity', 'Order Item Product Price', 'Sales',
    'Order Item Discount', 'Order Profit Per Order',
    'Customer City', 'Customer Country', 'Customer Segment', 'Customer State',
    'Product Name', 'Category Name', 'Department Name', 'Market',
    'order date (DateOrders)', 'shipping date (DateOrders)'
]

df = df_raw[KEEP_COLUMNS].copy()
print(f"Selected columns: {df.shape[1]}")
print(df.columns.tolist())

Selected columns: 25
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Delivery Status', 'Shipping Mode', 'Order Region', 'Order Status', 'Order Id', 'Order Item Id', 'Order Item Quantity', 'Order Item Product Price', 'Sales', 'Order Item Discount', 'Order Profit Per Order', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Product Name', 'Category Name', 'Department Name', 'Market', 'order date (DateOrders)', 'shipping date (DateOrders)']


In [6]:
df = df.rename(columns={
    'Type': 'payment_type',
    'Days for shipping (real)': 'days_shipping_real',
    'Days for shipment (scheduled)': 'days_shipping_scheduled',
    'Late_delivery_risk': 'late_delivery_risk',
    'Delivery Status': 'delivery_status',
    'Shipping Mode': 'shipping_mode',
    'Order Region': 'order_region',
    'Order Status': 'order_status',
    'Order Id': 'order_id',
    'Order Item Id': 'order_item_id',
    'Order Item Quantity': 'order_qty',
    'Order Item Product Price': 'unit_price',
    'Sales': 'sales',
    'Order Item Discount': 'discount',
    'Order Profit Per Order': 'profit',
    'Customer City': 'customer_city',
    'Customer Country': 'customer_country',
    'Customer Segment': 'customer_segment',
    'Customer State': 'customer_state',
    'Product Name': 'product_name',
    'Category Name': 'category_name',
    'Department Name': 'department_name',
    'Market': 'market',
    'order date (DateOrders)': 'order_date',
    'shipping date (DateOrders)': 'ship_date',
})

print("Columns renamed:")
print(df.columns.tolist())

Columns renamed:
['payment_type', 'days_shipping_real', 'days_shipping_scheduled', 'late_delivery_risk', 'delivery_status', 'shipping_mode', 'order_region', 'order_status', 'order_id', 'order_item_id', 'order_qty', 'unit_price', 'sales', 'discount', 'profit', 'customer_city', 'customer_country', 'customer_segment', 'customer_state', 'product_name', 'category_name', 'department_name', 'market', 'order_date', 'ship_date']


In [7]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date']  = pd.to_datetime(df['ship_date'])

numeric_cols = [
    'days_shipping_real', 'days_shipping_scheduled', 'late_delivery_risk',
    'order_id', 'order_item_id', 'order_qty',
    'unit_price', 'sales', 'discount', 'profit'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types fixed:")
print(df[['order_date', 'ship_date'] + numeric_cols].dtypes)

df['order_year']    = df['order_date'].dt.year
df['order_month']   = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter
df['order_dow']     = df['order_date'].dt.day_name()
df['is_full_year']  = df['order_year'].isin([2015, 2016, 2017]).astype(int)

print("\nDate parts added:")
print(df[['order_date', 'order_year', 'order_month', 'order_quarter', 'order_dow', 'is_full_year']].head())

Data types fixed:
order_date                 datetime64[us]
ship_date                  datetime64[us]
days_shipping_real                  int64
days_shipping_scheduled             int64
late_delivery_risk                  int64
order_id                            int64
order_item_id                       int64
order_qty                           int64
unit_price                        float64
sales                             float64
discount                          float64
profit                            float64
dtype: object

Date parts added:
           order_date  order_year  order_month  order_quarter  order_dow  \
0 2018-01-31 22:56:00        2018            1              1  Wednesday   
1 2018-01-13 12:27:00        2018            1              1   Saturday   
2 2018-01-13 12:06:00        2018            1              1   Saturday   
3 2018-01-13 11:45:00        2018            1              1   Saturday   
4 2018-01-13 11:24:00        2018            1              1   S

In [8]:

df['is_on_time'] = (
    df['delivery_status'].isin(['Advance shipping', 'Shipping on time'])
).astype(int)

# Order completion: was the order fully processed (not canceled/fraud)?
df['is_complete'] = (df['order_status'] == 'COMPLETE').astype(int)

df['is_qty_fulfilled'] = 1

# OTIF = On-Time AND In-Full
df['is_otif'] = ((df['is_on_time'] == 1) & (df['is_qty_fulfilled'] == 1)).astype(int)

# SLA breach: did actual shipping take longer than promised?
df['is_sla_breach'] = (df['days_shipping_real'] > df['days_shipping_scheduled']).astype(int)

# How many days late (or early) was it, on average?
df['days_variance'] = df['days_shipping_real'] - df['days_shipping_scheduled']

# Perfect order: on-time AND complete AND no SLA breach
df['is_perfect_order'] = (
    (df['is_on_time'] == 1) &
    (df['is_complete'] == 1) &
    (df['is_sla_breach'] == 0)
).astype(int)

print("KPI flags created:")
print("  → is_on_time")
print("  → is_complete")
print("  → is_qty_fulfilled")
print("  → is_otif")
print("  → is_sla_breach")
print("  → days_variance")
print("  → is_perfect_order")

print(f"\nOTD rate: {df['is_on_time'].mean()*100:.1f}%")
print(f"OTIF rate: {df['is_otif'].mean()*100:.1f}%")
print(f"SLA breach rate: {df['is_sla_breach'].mean()*100:.1f}%")
print(f"Perfect order rate: {df['is_perfect_order'].mean()*100:.1f}%")

KPI flags created:
  → is_on_time
  → is_complete
  → is_qty_fulfilled
  → is_otif
  → is_sla_breach
  → days_variance
  → is_perfect_order

OTD rate: 40.9%
OTIF rate: 40.9%
SLA breach rate: 57.3%
Perfect order rate: 14.0%


In [9]:

print(f"Final shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(df.columns.tolist())

Final shape: 180,519 rows, 37 columns
['payment_type', 'days_shipping_real', 'days_shipping_scheduled', 'late_delivery_risk', 'delivery_status', 'shipping_mode', 'order_region', 'order_status', 'order_id', 'order_item_id', 'order_qty', 'unit_price', 'sales', 'discount', 'profit', 'customer_city', 'customer_country', 'customer_segment', 'customer_state', 'product_name', 'category_name', 'department_name', 'market', 'order_date', 'ship_date', 'order_year', 'order_month', 'order_quarter', 'order_dow', 'is_full_year', 'is_on_time', 'is_complete', 'is_qty_fulfilled', 'is_otif', 'is_sla_breach', 'days_variance', 'is_perfect_order']


In [10]:

CLEAN_PATH = OUTPUT_PATH / "supplychain_clean.csv"

with open(CLEAN_PATH, "w", newline="", encoding="utf-8") as f:
    df.to_csv(f, index=False, lineterminator="\n")

print(f"Clean data exported to: {CLEAN_PATH}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

Clean data exported to: c:\Desktop\try and code\supply-chain-project\data\supplychain_clean.csv
Rows: 180,519 | Columns: 37


In [ ]:
df = df.rename(columns={
    'Type': 'payment_type',
    'Days for shipping (real)': 'days_shipping_real',
    'Days for shipment (scheduled)': 'days_shipping_scheduled',
    'Late_delivery_risk': 'late_delivery_risk',
    'Delivery Status': 'delivery_status',
    'Shipping Mode': 'shipping_mode',
    'Order Region': 'order_region',
    'Order Status': 'order_status',
    'Order Id': 'order_id',
    'Order Item Id': 'order_item_id',
    'Order Item Quantity': 'order_qty',
    'Order Item Product Price': 'unit_price',
    'Sales': 'sales',
    'Order Item Discount': 'discount',
    'Order Profit Per Order': 'profit',
    'Customer City': 'customer_city',
    'Customer Country': 'customer_country',
    'Customer Segment': 'customer_segment',
    'Customer State': 'customer_state',
    'Product Name': 'product_name',
    'Category Name': 'category_name',
    'Department Name': 'department_name',
    'Market': 'market',
    'order date (DateOrders)': 'order_date',
    'shipping date (DateOrders)': 'ship_date',
})

print("Columns renamed:")
print(df.columns.tolist())

Columns renamed:
['payment_type', 'days_shipping_real', 'days_shipping_scheduled', 'late_delivery_risk', 'delivery_status', 'shipping_mode', 'order_region', 'order_status', 'order_id', 'order_item_id', 'order_qty', 'unit_price', 'sales', 'discount', 'profit', 'customer_city', 'customer_country', 'customer_segment', 'customer_state', 'product_name', 'category_name', 'department_name', 'market', 'order_date', 'ship_date']
